In [178]:
import sys
print(sys.executable)

c:\Users\TUSHAR\Desktop\sentiment project\.venv\Scripts\python.exe


In [179]:
# Import Dependancy
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix,classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

In [180]:
df = pd.read_csv(
    "sentiment data/train.csv",
    encoding="latin-1"
)

df.head()


,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26


In [181]:
df.shape


(27481, 10)

In [182]:
df=df[['text','sentiment']]

In [183]:
df.head()

,text,sentiment
0,"I`d have responded, if I were going",neutral
1,Sooo SAD I will miss you here in San Diego!!!,negative
2,my boss is bullying me...,negative
3,what interview! leave me alone,negative
4,"Sons of ****, why couldn`t they put them on t...",negative


In [184]:
df.sentiment.value_counts()

sentiment
neutral     11118
positive     8582
negative     7781
Name: count, dtype: int64

In [185]:
df.isnull().sum()

text         1
sentiment    0
dtype: int64

In [186]:
df=df.dropna(subset=['text'])

In [187]:
print(df.isnull().sum())
print(df.shape)

text         0
sentiment    0
dtype: int64
(27480, 2)


In [188]:
df.duplicated().sum()

np.int64(0)

In [189]:
df['text'].head(10)

0                  I`d have responded, if I were going
1        Sooo SAD I will miss you here in San Diego!!!
2                            my boss is bullying me...
3                       what interview! leave me alone
4     Sons of ****, why couldn`t they put them on t...
5    http://www.dothebouncy.com/smf - some shameles...
6    2am feedings for the baby are fun when he is a...
7                                           Soooo high
8                                          Both of you
9     Journey!? Wow... u just became cooler.  hehe....
Name: text, dtype: str

## cleaning the data

In [190]:
import re
import emoji

def clean_text(text):
    text = str(text)
    
    # Convert emojis to text
    text = emoji.demojize(text, delimiters=(" ", " "))
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove @mentions
    text = re.sub(r'@\w+', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['clean_text'] = df['text'].apply(clean_text)

In [191]:
df[['text', 'clean_text']].head(10)

,text,clean_text
0,"I`d have responded, if I were going","i`d have responded, if i were going"
1,Sooo SAD I will miss you here in San Diego!!!,sooo sad i will miss you here in san diego!!!
2,my boss is bullying me...,my boss is bullying me...
3,what interview! leave me alone,what interview! leave me alone
4,"Sons of ****, why couldn`t they put them on t...","sons of ****, why couldn`t they put them on th..."
5,http://www.dothebouncy.com/smf - some shameles...,- some shameless plugging for the best rangers...
6,2am feedings for the baby are fun when he is a...,2am feedings for the baby are fun when he is a...
7,Soooo high,soooo high
8,Both of you,both of you
9,Journey!? Wow... u just became cooler. hehe....,journey!? wow... u just became cooler. hehe......


In [192]:
# ['sentiment'] = df['sentiment'].map({
#     'negative': 0,
#     'neutral': 1,
#     'positive': 2
# })

# df['sentiment'].value_counts()

## **train test split**

In [193]:
X = df['clean_text']
y = df['sentiment']

In [194]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(df['sentiment'])

In [195]:
print(encoder.classes_)
print(y_encoded[:20])

['negative' 'neutral' 'positive']
[1 0 0 0 0 1 2 1 1 2 1 2 0 0 1 0 0 0 0 1]


In [196]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [197]:
feature_extraction = TfidfVectorizer(min_df=1, stop_words='english', lowercase=True)


X_train = feature_extraction.fit_transform(X_train)
X_test = feature_extraction.transform(X_test)

In [198]:
print(X_train.shape)
print(X_test.shape)

(21984, 21523)
(5496, 21523)


## fitting model


In [199]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [200]:
test_pred = model.predict(X_test)
print(test_pred[:20])

[1 0 0 0 2 0 0 1 0 0 2 1 0 2 1 1 1 0 2 1]


In [201]:
accuracy = accuracy_score(y_test, test_pred)

print("Test Accuracy:", accuracy)

Test Accuracy: 0.6874090247452693


In [202]:
print(classification_report(
    y_test,
    test_pred,
    target_names=['Negative', 'Neutral', 'Positive']
))

              precision    recall  f1-score   support

    Negative       0.72      0.60      0.66      1556
     Neutral       0.62      0.75      0.68      2223
    Positive       0.78      0.68      0.73      1717

    accuracy                           0.69      5496
   macro avg       0.71      0.68      0.69      5496
weighted avg       0.70      0.69      0.69      5496



In [203]:
cm = confusion_matrix(y_test, test_pred)

print(cm)

[[ 940  535   81]
 [ 303 1663  257]
 [  59  483 1175]]


## checking wit no stopwords

In [204]:
X_text = df['clean_text']
y = df['sentiment']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [205]:
tfidf = TfidfVectorizer(
    lowercase=True
)

In [206]:
X_train_new = tfidf.fit_transform(X_train_text)

X_test_new = tfidf.transform(X_test_text)

In [207]:
model_new = LogisticRegression(max_iter=1000)

model_new.fit(X_train_new, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [208]:
test_pred_new = model_new.predict(X_test_new)

In [209]:
accuracy_new = accuracy_score(y_test, test_pred_new)

print("New Test Accuracy:", accuracy_new)

New Test Accuracy: 0.6846797671033479


In [210]:
print(classification_report(
    y_test,
    test_pred_new,
    target_names=['Negative', 'Neutral', 'Positive']
))

              precision    recall  f1-score   support

    Negative       0.71      0.61      0.66      1556
     Neutral       0.62      0.74      0.67      2223
    Positive       0.78      0.68      0.73      1717

    accuracy                           0.68      5496
   macro avg       0.70      0.68      0.69      5496
weighted avg       0.70      0.68      0.69      5496



## trying svm

In [211]:
from sklearn.svm import LinearSVC
svm_model = LinearSVC()

svm_model.fit(X_train_new, y_train)

svm_pred = svm_model.predict(X_test_new)

In [212]:
print("SVM Accuracy:", accuracy_score(y_test, svm_pred))

SVM Accuracy: 0.6752183406113537


## trying deep learning

In [213]:
import tensorflow as tf

print(tf.__version__)

2.21.0


In [214]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(
    num_words=20000,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train_text)

In [215]:
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

In [216]:
max_length = 100

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

In [217]:
print(X_train_pad.shape)
print(X_test_pad.shape)

(21984, 100)
(5496, 100)


## lstm model


In [218]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Bidirectional,
    Dense,
    Dropout
)

model_lstm = Sequential([
    Input(shape=(100,)),

    Embedding(
        input_dim=20000,
        output_dim=128,
        mask_zero=True
    ),

    Bidirectional(
        LSTM(64)
    ),

    Dropout(0.5),

    Dense(3, activation='softmax')
])

In [219]:
model_lstm.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [220]:
model_lstm.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 100, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,659,203 (10.14 MB)

 Trainable params: 2,659,203 (10.14 MB)

 Non-trainable params: 0 (0.00 B)

In [221]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
# history = model_lstm.fit(
#     X_train_pad,
#     y_train,
#     validation_split=0.1,
#     epochs=10,
#     batch_size=64,
#     callbacks=[early_stop]
# )

ValueError: Invalid dtype: str

In [ ]:
# test_loss, test_accuracy = model_lstm.evaluate(
#     X_test_pad,
#     y_test,
#     verbose=1
# )

# print("LSTM Test Accuracy:", test_accuracy)

ValueError: Invalid dtype: str

## testing on test data

In [226]:
test_df = pd.read_csv(
    "sentiment data/test.csv",
    encoding="latin-1"
)

test_df = test_df[['text', 'sentiment']]
test_df = test_df.dropna(subset=['text'])

In [227]:
print(test_df.shape)
print(test_df.columns)

(3534, 2)
Index(['text', 'sentiment'], dtype='str')


In [228]:
print(test_df['sentiment'].value_counts())

sentiment
neutral     1430
positive    1103
negative    1001
Name: count, dtype: int64


In [229]:
y_test_external = encoder.transform(test_df['sentiment'])

In [230]:
print(y_test_external[:20])

[1 2 0 2 2 2 0 0 1 1 0 1 1 0 0 0 0 0 2 1]


In [231]:
print(type(tfidf))

<class 'sklearn.feature_extraction.text.TfidfVectorizer'>


In [235]:
X_test_external = feature_extraction.transform(
    test_df['text']
)

print(X_test_external.shape)

(3534, 21523)


In [236]:
pred_external = model.predict(X_test_external)

In [237]:
print(
    "External Test Accuracy:",
    accuracy_score(y_test_external, pred_external)
)

External Test Accuracy: 0.6890209394453877


In [238]:
import joblib

In [240]:
from sklearn.pipeline import Pipeline
import joblib

final_model = Pipeline([
    ("tfidf", feature_extraction),
    ("classifier", model)
])

final_model.fit(X_train_text, y_train)

joblib.dump(final_model, "model.pkl")

['model.pkl']

In [5]:
import pandas as pd

big_df = pd.read_csv(
    "sentiment data/bigdata.csv",
    encoding="latin-1",
    nrows=5
)

print(big_df.shape)
print(big_df.columns)
print(big_df.head())

(5, 6)
Index(['polarity of tweet ', 'id of the tweet', 'date of the tweet', 'query',
       'user', 'text of the tweet '],
      dtype='str')
   polarity of tweet   id of the tweet             date of the tweet  \
0                   0       1467810672  Mon Apr 06 22:19:49 PDT 2009   
1                   0       1467810917  Mon Apr 06 22:19:53 PDT 2009   
2                   0       1467811184  Mon Apr 06 22:19:57 PDT 2009   
3                   0       1467811193  Mon Apr 06 22:19:57 PDT 2009   
4                   0       1467811372  Mon Apr 06 22:20:00 PDT 2009   

      query           user                                 text of the tweet   
0  NO_QUERY  scotthamilton  is upset that he can't update his Facebook by ...  
1  NO_QUERY       mattycus  @Kenichan I dived many times for the ball. Man...  
2  NO_QUERY        ElleCTF    my whole body feels itchy and like its on fire   
3  NO_QUERY         Karoli  @nationwideclass no, it's not behaving at all....  
4  NO_QUERY       joy_wol

In [7]:
import pandas as pd

big_sample = pd.read_csv(
    "sentiment data/bigdata.csv",
    encoding="latin-1",
    nrows=5
)

print(big_sample.columns.tolist())

['polarity of tweet\xa0', 'id of the tweet', 'date of the tweet', 'query', 'user', 'text of the tweet\xa0']


In [ ]:
big_sample = pd.read_csv(
    "sentiment data/bigdata.csv",
    encoding="latin-1",
    usecols=[0, 5],
    nrows=100000
)

big_sample.columns = ["sentiment", "text"]

print(big_sample.shape)
print(big_sample.head())
print(big_sample["sentiment"].value_counts())

(100000, 2)
   sentiment                                               text
0          0  is upset that he can't update his Facebook by ...
1          0  @Kenichan I dived many times for the ball. Man...
2          0    my whole body feels itchy and like its on fire 
3          0  @nationwideclass no, it's not behaving at all....
4          0                      @Kwesidei not the whole crew 
sentiment
0    100000
Name: count, dtype: int64


In [ ]:
# big_sample_2 = pd.read_csv(
#     "sentiment data/bigdata.csv",
#     encoding="latin-1",
#     usecols=[0, 5],
#     skiprows=500000,
#     nrows=100000
# )

# big_sample_2.columns = ["sentiment", "text"]

# print(big_sample_2["sentiment"].value_counts())

sentiment
0    100000
Name: count, dtype: int64


In [ ]:
# big_sample = pd.read_csv(
#     "sentiment data/bigdata.csv",
#     encoding="latin-1",
#     usecols=[0, 5],
#     skiprows=lambda x: x != 0 and x % 10000 != 0
# )

# big_sample.columns = ["sentiment", "text"]

# print(big_sample.shape)
# print(big_sample["sentiment"].value_counts())

(104, 2)
sentiment
0    79
4    25
Name: count, dtype: int64
